# ImpactSynth — synthetic CT from MR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fideus-labs/KonfAI/blob/main/examples/ImpactSynth/ImpactSynth_demo.ipynb)

**Run all the cells.** `impact-synth-konfai` is a KonfAI **app**: a published model on the Hugging Face Hub
([`VBoussot/ImpactSynth`](https://huggingface.co/VBoussot/ImpactSynth)) wrapped behind one command, for **synthetic CT** from MR or CBCT.
No training, no config to write — the app carries its own.

The model is downloaded on the first run (a few hundred MB) and a **GPU is strongly recommended**; on
Colab pick *Runtime > Change runtime type > GPU*.

In [ ]:
# Setup: find KonfAI (cloning it on Colab), install what is missing, load the notebook helpers.
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    REPO_DIR = Path("/content/KonfAI")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/fideus-labs/KonfAI", str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples").is_dir())
sys.path.insert(0, str(REPO_DIR / "examples"))

from konfai_demo import read, run, setup, show

EXAMPLE_DIR, DATASET_DIR, DEVICE = setup(REPO_DIR, "ImpactSynth", ("konfai", f"{REPO_DIR}[imaging]"), ("konfai_apps", str(REPO_DIR / "konfai-apps")), ("impact_synth_konfai", str(REPO_DIR / "apps" / "impact_synth")), "SimpleITK", "huggingface_hub", "matplotlib")

## 1. One public demo case

A single MR volume from the public demo dataset.

In [ ]:
import shutil

from huggingface_hub import snapshot_download

if not any(DATASET_DIR.glob("*/MR.mha")):
    snapshot_download("VBoussot/konfai-demo", repo_type="dataset", allow_patterns="Synthesis/**", local_dir=str(DATASET_DIR))
    for case in (DATASET_DIR / "Synthesis").iterdir():
        shutil.move(str(case), DATASET_DIR / case.name)
    shutil.rmtree(DATASET_DIR / "Synthesis")
    shutil.rmtree(DATASET_DIR / ".cache", ignore_errors=True)

CASES = sorted(path.name for path in DATASET_DIR.iterdir() if path.is_dir())
print(len(CASES), "cases:", ", ".join(CASES))

## 2. Look at the input MR

In [ ]:
import numpy as np

case = DATASET_DIR / CASES[0]
INPUT_PATH = case / "MR.mha"
image = read(INPUT_PATH).astype(np.float32)
print(case.name, "| shape:", image.shape)

low, high = np.percentile(image, [1, 99])


def window(index):
    """One slice, clipped to a robust intensity window so a few extreme voxels do not wash it out."""
    return np.clip(image[index], low, high)


MID = image.shape[0] // 2
show([(f"MR — slice {MID}", window(MID), "gray")])

## 3. Run the model

One command. KonfAI does the rest: it reads the volume lazily, runs the network patch by patch, and
reassembles the synthetic CT with overlap blending — never loading the full volume into memory.

```bash
impact-synth-konfai synthesize MR -i input.mha -o ./Output/ --gpu 0
```

In [ ]:
OUTPUT_DIR = EXAMPLE_DIR / "Output"

run("impact-synth-konfai", "synthesize", "MR", "-i", str(INPUT_PATH), "-o", str(OUTPUT_DIR), *DEVICE)
print("wrote:", *sorted(path.name for path in OUTPUT_DIR.rglob("*.mha")))

## 4. The result

The synthetic CT the app produced, on the input grid.

In [ ]:
import numpy as np

produced = sorted(OUTPUT_DIR.rglob("*.mha"))
result = read(produced[0])
print(produced[0].relative_to(OUTPUT_DIR), "|", result.shape, "| values:", np.unique(result)[:12], "...")

SLICE = MID
show([
    ("input MR", window(SLICE), "gray"),
    ("synthetic CT", result[SLICE], "gray", (-500.0, 1000.0)),
])

## What to change next

- **your own volume** — point `-i` at any `.mha` / `.nii.gz`, or at an OME-Zarr or DICOM directory;
  KonfAI detects the store format on read.
- **score it** — `impact-synth-konfai eval MR -i input.mha --gt reference.mha -o Output`.
- **everything at once** — `impact-synth-konfai pipeline ...` chains inference, evaluation and uncertainty.
- **see how an app is built** — `apps/impact_synth/` in this repo is the whole wrapper.